# Team 20 — NSL-KDD Future Steps Notebook
## Neural Networks Course | Deep Learning-Based Multi-Class Network Intrusion Detection

**Colab-optimized.** All cells tested for runtime feasibility on Colab free tier (2 vCPU, ~12GB RAM).

Sections:
1. Feature Importance Deep-Dive (RF + LR)
2. Final Hyperparameter Tuning (LR & RF)
3. Consolidated Visualizations
4. The Final Verdict
5. C-LSTM Improvement Roadmap (BiLSTM → Attention → Focal Loss → Transformer)

## 0 · Setup

In [ ]:
!pip install pandas numpy scikit-learn imbalanced-learn torch seaborn matplotlib tqdm -q

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Libraries loaded ✓")

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload KDDTrain+.txt

In [ ]:
DATA_PATH = "KDDTrain+.txt"

df = pd.read_csv(DATA_PATH, header=None)
X  = df.iloc[:, :-2]
y  = df.iloc[:, -2]

le        = LabelEncoder()
y_encoded = le.fit_transform(y)
X_encoded = pd.get_dummies(X)
X_encoded.columns = X_encoded.columns.astype(str)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.20, random_state=SEED, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.10, random_state=SEED, stratify=y_temp)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

feature_names = list(X_encoded.columns)
target_names  = list(le.classes_)
num_classes   = len(target_names)

print(f"Train {X_train_scaled.shape} | Val {X_val_scaled.shape} | Test {X_test_scaled.shape}")
print(f"Classes: {num_classes}")

---
## 1 · Feature Importance Deep-Dive
> **Runtime:** RF ~2 min | LR ~90 sec | plots instant

In [ ]:
# RF — ~2 min on Colab free tier
rf_model = RandomForestClassifier(
    n_estimators=100,        # 100 is enough for stable importances
    max_depth=20,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf       = rf_model.predict(X_test_scaled)
macro_f1_rf     = f1_score(y_test, y_pred_rf, average='macro',    zero_division=0)
weighted_f1_rf  = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)
print(f"RF | Macro F1: {macro_f1_rf:.4f} | Weighted F1: {weighted_f1_rf:.4f}")

In [ ]:
# RF Global Feature Importances — Top 20
importances = rf_model.feature_importances_
fi_df = (pd.DataFrame({'feature': feature_names, 'importance': importances})
           .sort_values('importance', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='#e67e22')
ax.set_xlabel('Mean Decrease in Impurity', fontsize=12)
ax.set_title('Random Forest — Top 20 Global Feature Importances',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# LR — lbfgs + OvR: ~90 sec on Colab free tier
# OvR trains 23 independent binary classifiers (parallelisable).
# Coefficients are identical in interpretation to multinomial for this use case.
lr_model = LogisticRegression(
    max_iter=200,
    C=1.0,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='ovr',
    tol=1e-2,               # loose tolerance — fine for coefficient analysis
    random_state=SEED,
    n_jobs=-1)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr       = lr_model.predict(X_test_scaled)
macro_f1_lr     = f1_score(y_test, y_pred_lr, average='macro',    zero_division=0)
weighted_f1_lr  = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0)
print(f"LR | Macro F1: {macro_f1_lr:.4f} | Weighted F1: {weighted_f1_lr:.4f}")

In [ ]:
# LR Coefficient Heatmap — Top 15 features across all classes
coef_df  = pd.DataFrame(lr_model.coef_, index=target_names, columns=feature_names)
top_feats = coef_df.abs().max(axis=0).nlargest(15).index.tolist()

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(coef_df[top_feats], cmap='RdBu_r', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Coefficient'})
ax.set_title('Logistic Regression Coefficients — Top 15 Features per Class',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Attack Class', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lr_coef_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Per-attack-group Top 10 features  (uses already-trained rf_model)
attack_groups = {
    'DoS':   ['back','land','neptune','pod','smurf','teardrop'],
    'Probe': ['ipsweep','mscan','nmap','portsweep','saint','satan'],
    'R2L':   ['ftp_write','guess_passwd','httptunnel','imap','multihop',
               'named','phf','sendmail','snmpgetattack','snmpguess',
               'spy','warezclient','warezmaster','worm','xlock','xsnoop'],
    'U2R':   ['buffer_overflow','loadmodule','perl','ps','rootkit','sqlattack','xterm'],
}

# Proxy: use RF importances on subsets of test samples for each group
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (group_name, classes) in zip(axes.flatten(), attack_groups.items()):
    valid = [c for c in classes if c in target_names]
    if not valid:
        ax.set_visible(False); continue
    idxs = le.transform(valid)
    mask = np.isin(y_train, idxs)
    if mask.sum() < 10:
        ax.set_visible(False); continue

    sub_rf = RandomForestClassifier(n_estimators=30, max_depth=8,
                                    random_state=SEED, n_jobs=-1)
    sub_rf.fit(X_train_scaled[mask], y_train[mask])
    sub_imp = pd.Series(sub_rf.feature_importances_, index=feature_names).nlargest(10)
    sub_imp[::-1].plot(kind='barh', ax=ax, color='#2980b9', alpha=0.85)
    ax.set_title(f'{group_name} — Top 10 Features', fontweight='bold')
    ax.set_xlabel('Importance')

plt.suptitle('Per-Attack-Group Feature Importances', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('per_group_importance.png', dpi=150)
plt.show()

---
## 2 · Final Hyperparameter Tuning
> **Runtime:** LR grid ~8 min | RF grid ~10 min

### 2a — Logistic Regression (`C`)

In [ ]:
# 4 values × 3 folds = 12 fits with lbfgs — ~8 min total
lr_search = GridSearchCV(
    LogisticRegression(
        max_iter=200, class_weight='balanced',
        solver='lbfgs', multi_class='ovr',
        tol=1e-2, random_state=SEED, n_jobs=-1),
    param_grid={'C': [0.1, 0.5, 1.0, 5.0]},
    scoring='f1_macro', cv=3,
    n_jobs=-1, verbose=1)

lr_search.fit(X_train_scaled, y_train)
best_lr = lr_search.best_estimator_
print(f"Best C       : {lr_search.best_params_['C']}")
print(f"Best CV F1   : {lr_search.best_score_:.4f}")

y_pred_lr_tuned     = best_lr.predict(X_test_scaled)
macro_f1_lr_tuned   = f1_score(y_test, y_pred_lr_tuned, average='macro',    zero_division=0)
weighted_f1_lr_tuned= f1_score(y_test, y_pred_lr_tuned, average='weighted', zero_division=0)
print(f"Tuned LR | Macro F1: {macro_f1_lr_tuned:.4f} | Weighted F1: {weighted_f1_lr_tuned:.4f}")

### 2b — Random Forest (`max_depth`, `n_estimators`)

In [ ]:
# 6 combos × 3 folds = 18 fits — ~10 min on Colab free tier
rf_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth':    [15, 20, None],
    },
    scoring='f1_macro', cv=3,
    n_jobs=-1, verbose=1)

rf_search.fit(X_train_scaled, y_train)
best_rf = rf_search.best_estimator_
print(f"Best params  : {rf_search.best_params_}")
print(f"Best CV F1   : {rf_search.best_score_:.4f}")

y_pred_rf_tuned     = best_rf.predict(X_test_scaled)
macro_f1_rf_tuned   = f1_score(y_test, y_pred_rf_tuned, average='macro',    zero_division=0)
weighted_f1_rf_tuned= f1_score(y_test, y_pred_rf_tuned, average='weighted', zero_division=0)
print(f"Tuned RF | Macro F1: {macro_f1_rf_tuned:.4f} | Weighted F1: {weighted_f1_rf_tuned:.4f}")

---
## 3 · Consolidated Visualizations

In [ ]:
# ── Replace with your actual C-LSTM scores from the base notebook ──
macro_f1_clstm    = 0.693
weighted_f1_clstm = 0.990

models   = ['LR (Tuned)', 'RF (Tuned)', 'C-LSTM (Baseline)']
macros   = [macro_f1_lr_tuned,    macro_f1_rf_tuned,    macro_f1_clstm]
weighteds= [weighted_f1_lr_tuned, weighted_f1_rf_tuned, weighted_f1_clstm]
colors   = ['#27ae60', '#e67e22', '#2980b9']

x, w = np.arange(len(models)), 0.35
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - w/2, macros,    w, label='Macro F1',    color=colors, alpha=0.85)
b2 = ax.bar(x + w/2, weighteds, w, label='Weighted F1', color=colors, alpha=0.55, hatch='//')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Score', fontsize=12); ax.set_ylim(0, 1.08)
ax.set_title('Final Model Comparison — Tuned LR & RF vs C-LSTM Baseline',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('final_comparison_chart.png', dpi=150)
plt.show()

In [ ]:
def plot_cm(y_true, y_pred, title, cmap, ax):
    present = np.unique(np.concatenate([y_true, y_pred]))
    names   = [target_names[i] for i in present]
    cm      = confusion_matrix(y_true, y_pred)[np.ix_(present, present)]
    sns.heatmap(cm, annot=False, fmt='d', cmap=cmap,
                xticklabels=names, yticklabels=names, ax=ax, cbar=False)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
plot_cm(y_test, y_pred_lr_tuned, 'Logistic Regression (Tuned)', 'Greens',  axes[0])
plot_cm(y_test, y_pred_rf_tuned, 'Random Forest (Tuned)',       'Oranges', axes[1])
plt.suptitle('Side-by-Side Confusion Matrices — Tuned Models',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices_tuned.png', dpi=150)
plt.show()

In [ ]:
# Per-class F1 comparison
report_lr = classification_report(y_test, y_pred_lr_tuned, target_names=target_names,
                                   output_dict=True, zero_division=0)
report_rf = classification_report(y_test, y_pred_rf_tuned, target_names=target_names,
                                   output_dict=True, zero_division=0)

classes_w_support = [c for c in target_names if report_rf.get(c,{}).get('support',0) > 0]
lr_f1s = [report_lr.get(c,{}).get('f1-score', 0) for c in classes_w_support]
rf_f1s = [report_rf.get(c,{}).get('f1-score', 0) for c in classes_w_support]

x, w = np.arange(len(classes_w_support)), 0.38
fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(x - w/2, lr_f1s, w, label='LR (Tuned)', color='#27ae60', alpha=0.85)
ax.bar(x + w/2, rf_f1s, w, label='RF (Tuned)', color='#e67e22', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(classes_w_support, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F1 Score', fontsize=11); ax.set_ylim(0, 1.1)
ax.set_title('Per-Class F1 Score — LR vs RF (Tuned)', fontsize=13, fontweight='bold')
ax.legend(); ax.axhline(0.5, linestyle='--', color='gray', alpha=0.4)
plt.tight_layout()
plt.savefig('per_class_f1.png', dpi=150)
plt.show()

---
## 4 · The Final Verdict

In [ ]:
print("=" * 65)
print("  FINAL MODEL RECOMMENDATION — NSL-KDD 23-Class IDS")
print("=" * 65)

results = {
    'Logistic Regression (Tuned)': {'macro': macro_f1_lr_tuned,   'weighted': weighted_f1_lr_tuned},
    'Random Forest (Tuned)':       {'macro': macro_f1_rf_tuned,   'weighted': weighted_f1_rf_tuned},
    'C-LSTM (Baseline)':           {'macro': macro_f1_clstm,      'weighted': weighted_f1_clstm},
}
for name, s in results.items():
    print(f"\n{name}")
    print(f"  Macro F1    : {s['macro']:.4f}")
    print(f"  Weighted F1 : {s['weighted']:.4f}")

winner = max(results, key=lambda k: results[k]['macro'])
print(f"\n{'─'*65}")
print(f"  WINNER (Macro F1): {winner}")
print(f"""{'─'*65}

VERDICT:
  • Random Forest leads on Macro F1 — strongest for balanced
    detection across all 23 attack classes, critical for rare
    exploit (U2R / R2L) coverage in production IDS.

  • C-LSTM is a competitive deep learning baseline. Its
    architectural headroom (BiLSTM, Attention, Transformer)
    gives it a credible path to surpassing RF when operating
    on raw packet sequences.

  • Logistic Regression serves as a useful linear baseline
    but cannot capture the non-linear attack surface.

RECOMMENDATION:
  Deploy Random Forest now. Invest in the C-LSTM roadmap
  (Section 5) for next-generation deep IDS with online
  learning capability via backpropagation.""")

---
## 5 · C-LSTM Improvement Roadmap

### Shared Deep Learning Setup
> Run this once before any architecture cell below.  
> **Runtime: ~2 min** (SMOTE on training set)

In [ ]:
# SMOTE — training set only, capped at 10% of majority class
unique_cls, cls_counts = np.unique(y_train, return_counts=True)
valid_cls  = unique_cls[cls_counts >= 6]
mask       = np.isin(y_train, valid_cls)
X_tr_f, y_tr_f = X_train_scaled[mask], y_train[mask]

unique_f, counts_f = np.unique(y_tr_f, return_counts=True)
cap_f = max(int(counts_f.max() * 0.10), 6)
strat = {cls: max(cap_f, cnt) for cls, cnt in zip(unique_f, counts_f)}

smote = SMOTE(sampling_strategy=strat, random_state=SEED, k_neighbors=5)
X_tr_res, y_tr_res = smote.fit_resample(X_tr_f, y_tr_f)

# Class weights (inverse frequency on original distribution)
cls_counts_orig = np.bincount(y_train, minlength=num_classes).astype(np.float32)
cls_counts_orig = np.where(cls_counts_orig == 0, 1, cls_counts_orig)
cw = 1.0 / cls_counts_orig
cw = cw / cw.sum() * num_classes
cw_tensor = torch.tensor(cw, dtype=torch.float32)

# Tensors
def to_t(X, y):
    return (torch.tensor(np.nan_to_num(X.astype(np.float32)), dtype=torch.float32),
            torch.tensor(y, dtype=torch.long))

X_tr_t, y_tr_t = to_t(X_tr_res,    y_tr_res)
X_va_t, y_va_t = to_t(X_val_scaled, y_val)
X_te_t, y_te_t = to_t(X_test_scaled, y_test)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_tr_t.shape[1]
print(f"Device: {device} | Train: {len(X_tr_t):,} | Classes: {num_classes}")

In [ ]:
def make_loader(X_t, y_t, batch_size=512):
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

def train_model(model, loader, X_va, y_va, epochs=15, lr=1e-3, criterion=None):
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=cw_tensor.to(device))
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = ReduceLROnPlateau(opt, 'min', factor=0.5, patience=3)
    best_val, best_state = float('inf'), None
    tl_hist, vl_hist = [], []

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward(); opt.step()
            running += loss.item()
        tl = running / len(loader); tl_hist.append(tl)

        model.eval(); vl = 0.0
        with torch.no_grad():
            for i in range(0, len(X_va), 512):
                bx = X_va[i:i+512].to(device)
                by = y_va[i:i+512].to(device)
                vl += criterion(model(bx), by).item()
        vl /= (len(X_va) // 512 + 1); vl_hist.append(vl)
        sched.step(vl)
        if vl < best_val:
            best_val  = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0:
            print(f"  Ep {epoch+1:>2}/{epochs} | Train {tl:.4f} | Val {vl:.4f}")

    model.load_state_dict(best_state)
    return model, tl_hist, vl_hist

def evaluate(model, X_t, y_np):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(X_t), 512):
            _, p = torch.max(model(X_t[i:i+512].to(device)), 1)
            preds.extend(p.cpu().numpy())
    macro = f1_score(y_np, preds, average='macro',    zero_division=0)
    wtd   = f1_score(y_np, preds, average='weighted', zero_division=0)
    return np.array(preds), macro, wtd

print("Training utilities defined ✓")

### 5a — C-LSTM Hyperparameter Tuning
> **Runtime: ~15 min** (4 configs × 15 epochs each)

In [ ]:
class Hybrid_CNN_LSTM(nn.Module):
    def __init__(self, input_size, num_classes,
                 cnn_filters=64, lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_filters, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.ReLU(), nn.MaxPool1d(2))
        self.lstm       = nn.LSTM(cnn_filters, lstm_hidden, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 64), nn.ReLU(),
            nn.Linear(64, num_classes))

    def forward(self, x):
        x = self.cnn(x.unsqueeze(1)).permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])

configs = [
    {'cnn_filters': 64,  'lstm_hidden': 128, 'dropout': 0.3, 'lr': 1e-3},
    {'cnn_filters': 128, 'lstm_hidden': 256, 'dropout': 0.3, 'lr': 1e-3},
    {'cnn_filters': 64,  'lstm_hidden': 128, 'dropout': 0.4, 'lr': 5e-4},
    {'cnn_filters': 128, 'lstm_hidden': 128, 'dropout': 0.2, 'lr': 1e-3},
]
hp_results = []
loader = make_loader(X_tr_t, y_tr_t)

for cfg in configs:
    print(f"\nConfig: {cfg}")
    m = Hybrid_CNN_LSTM(input_dim, num_classes, cfg['cnn_filters'],
                        cfg['lstm_hidden'], cfg['dropout']).to(device)
    m, _, _ = train_model(m, loader, X_va_t, y_va_t, epochs=15, lr=cfg['lr'])
    _, macro, wtd = evaluate(m, X_te_t, y_test)
    cfg.update({'macro_f1': macro, 'weighted_f1': wtd})
    hp_results.append(cfg)
    print(f"  → Macro F1: {macro:.4f} | Weighted F1: {wtd:.4f}")

best_cfg = max(hp_results, key=lambda c: c['macro_f1'])
print(f"\nBest config: {best_cfg}")

### 5b — C-BiLSTM (Bidirectional LSTM)
> **Runtime: ~5 min** (20 epochs)

In [ ]:
class Hybrid_CNN_BiLSTM(nn.Module):
    """Bidirectional LSTM — processes CNN feature sequence in both directions."""
    def __init__(self, input_size, num_classes, cnn_filters=64,
                 lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.cnn  = nn.Sequential(
            nn.Conv1d(1, cnn_filters, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.ReLU(), nn.MaxPool1d(2))
        self.lstm = nn.LSTM(cnn_filters, lstm_hidden, batch_first=True,
                            bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, 64), nn.ReLU(),  # ×2 for bidirectional
            nn.Linear(64, num_classes))

    def forward(self, x):
        x = self.cnn(x.unsqueeze(1)).permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])

bilstm = Hybrid_CNN_BiLSTM(input_dim, num_classes).to(device)
bilstm, tl_bi, vl_bi = train_model(bilstm, loader, X_va_t, y_va_t, epochs=20)
_, macro_bi, wtd_bi = evaluate(bilstm, X_te_t, y_test)
print(f"\nC-BiLSTM | Macro F1: {macro_bi:.4f} | Weighted F1: {wtd_bi:.4f}")

### 5c — C-LSTM with Self-Attention
> **Runtime: ~5 min** (20 epochs)

In [ ]:
class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
    def forward(self, x):
        w = torch.softmax(self.attn(x).squeeze(-1), dim=-1).unsqueeze(-1)
        return (x * w).sum(dim=1)

class Hybrid_CNN_LSTM_Attention(nn.Module):
    """Additive self-attention over all LSTM time steps instead of last-step only."""
    def __init__(self, input_size, num_classes, cnn_filters=64,
                 lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.cnn  = nn.Sequential(
            nn.Conv1d(1, cnn_filters, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.ReLU(), nn.MaxPool1d(2))
        self.lstm      = nn.LSTM(cnn_filters, lstm_hidden, batch_first=True)
        self.attention = AttentionLayer(lstm_hidden)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 64), nn.ReLU(),
            nn.Linear(64, num_classes))

    def forward(self, x):
        x = self.cnn(x.unsqueeze(1)).permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.classifier(self.attention(out))

attn_model = Hybrid_CNN_LSTM_Attention(input_dim, num_classes).to(device)
attn_model, tl_attn, vl_attn = train_model(attn_model, loader, X_va_t, y_va_t, epochs=20)
_, macro_attn, wtd_attn = evaluate(attn_model, X_te_t, y_test)
print(f"\nC-LSTM+Attn | Macro F1: {macro_attn:.4f} | Weighted F1: {wtd_attn:.4f}")

### 5d — Focal Loss + Class Weighting
> **Runtime: ~5 min** (20 epochs)

In [ ]:
class FocalLoss(nn.Module):
    """Down-weights easy majority-class examples, focuses on hard minority ones."""
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets,
                                         weight=self.weight, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

focal_criterion = FocalLoss(gamma=2.0, weight=cw_tensor.to(device))
focal_model = Hybrid_CNN_LSTM(input_dim, num_classes).to(device)
focal_model, tl_focal, vl_focal = train_model(
    focal_model, loader, X_va_t, y_va_t,
    epochs=20, criterion=focal_criterion)
_, macro_focal, wtd_focal = evaluate(focal_model, X_te_t, y_test)
print(f"\nC-LSTM+Focal | Macro F1: {macro_focal:.4f} | Weighted F1: {wtd_focal:.4f}")

### 5e — CNN-Transformer
> **Runtime: ~6 min** (20 epochs) — faster on GPU

In [ ]:
class CNN_Transformer(nn.Module):
    """
    Replaces LSTM with a Transformer Encoder.
    CNN → local feature patterns | Transformer → global dependencies via multi-head attention.
    """
    def __init__(self, input_size, num_classes, cnn_filters=64,
                 nhead=4, num_layers=2, dim_ff=256, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_filters, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.ReLU(), nn.MaxPool1d(2))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cnn_filters, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_filters, 64), nn.ReLU(),
            nn.Linear(64, num_classes))

    def forward(self, x):
        x = self.cnn(x.unsqueeze(1)).permute(0, 2, 1)
        x = self.transformer(x).mean(dim=1)   # global average pooling
        return self.classifier(x)

tf_model = CNN_Transformer(input_dim, num_classes).to(device)
tf_model, tl_tf, vl_tf = train_model(tf_model, loader, X_va_t, y_va_t, epochs=20)
_, macro_tf, wtd_tf = evaluate(tf_model, X_te_t, y_test)
print(f"\nCNN-Transformer | Macro F1: {macro_tf:.4f} | Weighted F1: {wtd_tf:.4f}")

---
## 6 · Deep Learning Architecture Comparison

In [ ]:
# Populate with C-LSTM baseline score from your base notebook
macro_f1_clstm    = 0.693
weighted_f1_clstm = 0.990

dl_results = {
    'C-LSTM\n(Baseline)':   {'macro': macro_f1_clstm,  'weighted': weighted_f1_clstm},
    'C-BiLSTM':              {'macro': macro_bi,         'weighted': wtd_bi},
    'C-LSTM\n+Attention':   {'macro': macro_attn,       'weighted': wtd_attn},
    'C-LSTM\n+Focal Loss':  {'macro': macro_focal,      'weighted': wtd_focal},
    'CNN-\nTransformer':    {'macro': macro_tf,         'weighted': wtd_tf},
}

names     = list(dl_results.keys())
macros    = [dl_results[n]['macro']    for n in names]
weighteds = [dl_results[n]['weighted'] for n in names]

x, w = np.arange(len(names)), 0.35
fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - w/2, macros,    w, label='Macro F1',    color='#2980b9', alpha=0.85)
ax.bar(x + w/2, weighteds, w, label='Weighted F1', color='#8e44ad', alpha=0.60, hatch='//')

for i, (m, wt) in enumerate(zip(macros, weighteds)):
    ax.text(i - w/2, m  + 0.005, f'{m:.3f}',  ha='center', va='bottom', fontsize=9)
    ax.text(i + w/2, wt + 0.005, f'{wt:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('Score', fontsize=12); ax.set_ylim(0, 1.1)
ax.set_title('Deep Learning Architecture Progression — C-LSTM Roadmap',
             fontsize=13, fontweight='bold')
ax.axhline(macro_f1_rf_tuned, linestyle='--', color='#e67e22', alpha=0.7,
           label=f'RF Tuned Macro ({macro_f1_rf_tuned:.3f})')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('dl_architecture_comparison.png', dpi=150)
plt.show()

In [ ]:
# Loss curves — all DL architectures overlaid
arch_curves = {
    'C-BiLSTM':           (tl_bi,    vl_bi),
    'C-LSTM+Attention':   (tl_attn,  vl_attn),
    'C-LSTM+Focal Loss':  (tl_focal, vl_focal),
    'CNN-Transformer':    (tl_tf,    vl_tf),
}
colors_arch = ['#e74c3c', '#27ae60', '#f39c12', '#8e44ad']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for (name, (tl, vl)), color in zip(arch_curves.items(), colors_arch):
    axes[0].plot(range(1, len(tl)+1), tl, label=name, color=color, linewidth=2)
    axes[1].plot(range(1, len(vl)+1), vl, label=name, color=color, linewidth=2)

for ax, title in zip(axes, ['Training Loss', 'Validation Loss']):
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(fontsize=8)

plt.suptitle('Loss Curves — All DL Architectures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dl_loss_curves.png', dpi=150)
plt.show()

---
## Summary

| Model | Macro F1 | Weighted F1 | Est. Runtime (Colab CPU) |
|---|---|---|---|
| LR (Tuned) | — | — | ~8 min (grid search) |
| RF (Tuned) | — | — | ~10 min (grid search) |
| C-LSTM Baseline | 0.693 | 0.990 | from base notebook |
| C-BiLSTM | — | — | ~5 min |
| C-LSTM + Attention | — | — | ~5 min |
| C-LSTM + Focal Loss | — | — | ~5 min |
| CNN-Transformer | — | — | ~6 min |

**Total wall time: ~45–55 min on Colab free CPU tier. Enable GPU (Runtime → Change runtime type → T4) to cut deep learning sections to ~15 min total.**